In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             1544795
   Errors:                    678
   Search Artists:            1803604
   Known Summary IDs:         1455842


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [50]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  59209
#   Artist Names To Get:  42659
#   Artist Names To Get:  31709

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   1782095
#   Artist Names To Get:  21509


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "9:50am")
tt = TermTime("today", "10:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-07-08 09:53:04
========================= termTime(day=today,time=10:00pm) =========================
   ====> Terminate Time Set To 2022-07-08 22:00:00 <====
   ====> Will Terminate Process 12 Hours and 6 Minutes From Now
Getting Sarah Rosen (0002470340)                          True
Getting Sarah Ryan (0000985350)                           True
Getting Marcipan (0000678768)                             True
Getting Robert "Marzipan" Green (0002617655)              True
Getting Sarah Reed (0000339802)                           True
Getting Sarah Reed (0001264509)                           True
Getting Sarah Goslee Reed (0000295618)                    True
Getting Sarah Wright (0000295533)                         True
Getting Sarah Reid (0001336831)                           True
Getting Sarah Rudd (0003242511)                           True
Getting Sarah Roddy (0004024299)                          True
Getting Sarah Bealby-Wr

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Jackie Marsala (0001557026)                       True
Getting Beatrice Marsala (0001764639)                     True
Getting Andy Marsala (0003291948)                         True
Getting William Marsala (0003851252)                      True
Getting Davide Marsala (0004006659)                       True
Getting Mr. Flowers (0000507232)                          True
Getting Mr. Flowers (0001991492)                          True
Getting Mars Fellows (0002085565)                         True
Getting Maureen N. McLane (0002388826)                    True
Getting Sarah E. Henneghan (0001847170)                   True
Getting Sarah E. Soulsby (0002274093)                     True
Getting Sarah E. Palmer (0002709536)                      True
Getting Sarah E. Newhall (0003022313)                     True
Getting Naara e Sarah (0004080865)                        True
Getting Marilinda Garcia (0002977959)                     True
Getting Mirlande (0001435792)                          

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Maurice Ulmer Group (0002895162)                  True
Getting Sarah Trevino (0002725391)                        True
Getting Sarah Draffen (0003288097)                        True
Getting Sarah Stevens-Estabrook (0001678059)              True
Getting Sarah Stephens (0001802891)                       True
Getting Sarah Stefan (0002397760)                         True
Getting Cyrus Stevens (0000600455)                        True
Getting Sara-Jean Stevens (0003177769)                    True
Getting Marry Sanders (0003697320)                        True
Getting Sarah Shafey (0001044623)                         True
Getting Sarah Shafey (0002131527)                         True
Getting Sarah Chaffee (0002876904)                        True
Getting Sarah Sitkin  (0002432381)                        True
Getting Sarah Sitken (0003725160)                         True
Getting Sarah Send (0001342418)                           True
Getting Sarah Sand (0002859699)                        

...

Getting Ahmed Sadaoui (0002969760)                        True
Getting Sid Ahmed (0000044938)                            True
Getting Said Ahmed (0002748928)                           True
Getting Ahmed & Souad (0000533086)                        True
Getting Sid Ahmed Khazradji (0001451972)                  True
Getting Sid Ahmed Ould (0001537595)                       True
Getting Sid Ahmed Agoumi (0002196549)                     True
Getting Sidi Ahmed Taanoun (0003490280)                   True
Getting Kai Svestad (0002651922)                          True
Getting Anton Mullan (0000442437)                         True
Getting Anton Mullan (0001655254)                         True
Getting Mark Kenyon (0001515098)                          True
Getting Bronce (0000625280)                               True
Getting Bronce (0001282907)                               True
Getting Bronce de Apodaca (0001251501)                    True
Getting Los Gitanillos de Bronce (0001260060)          

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Mark Janku (0002145671)                           True
Getting Mark Janiga (0002187083)                          True
Getting Mark Jenks (0002494508)                           True
Getting Apriskah (0003917127)                             True
Getting Mark Josher (0003568167)                          True
Getting Mark Larson (0001270049)                          True
Getting Mark Larson (0003953754)                          True
Getting Marcus Larson (0002185267)                        True
Getting Mark Larsen (0001476615)                          True
Getting Mark Lerson (0001816553)                          True
Getting Mark J. Larsen (0002960105)                       True
Getting Mark LaFalce (0001800219)                         True
Getting Mark Loveless (0003177093)                        True
Getting Mark Livolsi (0004117865)                         True
Getting Mark Kreider (0001070076)                         True
Getting Mark Kreider (0001722910)                      

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Apriel Dunlop (0001761528)                        True
Getting Dunlop (0003899235)                               

In [43]:
del searchedForErrors['0003281182']
errors.save(data=searchedForErrors)

In [ ]:
from lib.allmusic import moveLocalFiles
moveLocalFiles()

In [ ]:
tmp = []
for line in tmp:
    if line.startswith("Getting"):
        artistID = line.split()[-2][1:-1]
        print(artistID)
        del searchedForErrors[artistID]

# Create Tabs Data

In [ ]:
def getTabs(rData):
    extraData = rData.profile.extra
    tabs = extraData.get('tabs', []) if isinstance(extraData,dict) else []
    retval = {tab.title: tab.href for tab in tabs}
    return retval

In [ ]:
mio   = allmusic.MusicDBIO()
tabsData = None
for modVal in range(100):
    modValTabsData = mio.data.getModValData(modVal).apply(getTabs)
    tabsData = concat([tabsData, modValTabsData]) if isinstance(tabsData,Series) else modValTabsData

# Download Artist Discography Data

# Parse

In [ ]:
from utils import PoolIO
pio = PoolIO("AllMusic")
pio.parse(force=True)
pio.meta()
pio.sum()
pio.search()